# Air Quality EDA & Analysis

This single notebook combines all the steps of our project without losing any code: data loading, data cleaning, exploratory analysis (with statistics), and data visualization.

## 1. Data Loading

This code matches `Python/data_loading.py` exactly.

In [ ]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../Dataset/dataset.csv")
print("Raw shape:", df.shape)

# Display first 10 rows
print(df.head(10))

# Display info about the dataset
df.info()

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Basic counts
print("States  :", df["state"].nunique())
print("Cities  :", df["city"].nunique())
print("Stations:", df["station"].nunique())

# Check pollutant types
print("Pollutant value counts:")
print(df["pollutant_id"].value_counts())

# Check states
print("State value counts:")
print(df["state"].value_counts())

# Basic descriptive statistics
print("Summary stats for min, max, avg:")
print(df[["pollutant_min", "pollutant_max", "pollutant_avg"]].describe())

# Check for duplicates
print("Duplicate rows:", df.duplicated().sum())

## 2. Data Cleaning

This code matches `Python/data_cleaning.py` exactly.

In [ ]:
import pandas as pd

df = pd.read_csv("../Dataset/dataset.csv")
print("Raw shape:", df.shape)

# Drop country column - it's just 'India' everywhere
if "country" in df.columns:
    df = df.drop(columns=["country"])
print("Columns after dropping country:", df.columns.tolist())

# Fix state and city names - remove underscores, apply title case
df["state"] = df["state"].str.replace("_", " ").str.strip().str.title()
df["city"]  = df["city"].str.strip().str.title()
df["station"] = df["station"].str.strip()

print("Cleaned states list:")
print(sorted(df["state"].unique()))

# Convert last_update to datetime
df["last_update"] = pd.to_datetime(df["last_update"], dayfirst=True)
print("last_update type:", df["last_update"].dtype)

# Rows where min > max (data entry issues)
bad = df[df["pollutant_min"] > df["pollutant_max"]]
print("Bad rows (min > max):", len(bad))

# Drop rows where pollutant_avg is missing
before = len(df)
df = df.dropna(subset=["pollutant_avg"])
print("Rows removed with missing avg:", before - len(df))
print("Shape after cleaning:", df.shape)

df = df.reset_index(drop=True)

# Save the cleaned dataset
df.to_csv("../Dataset/cleaned dataset.csv", index=False)
print("Saved to '../Dataset/cleaned dataset.csv'")

## 3. Exploratory Analysis & Statistics

This code matches `Python/exploratory_analysis.py` exactly.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../Dataset/cleaned dataset.csv")
print("Dataset shape:", df.shape)

print("=== Summary Statistics ===")
print(df.describe())

print("\n=== Average Pollutant Values ===")
print(df.groupby("pollutant_id")["pollutant_avg"].mean().round(2))

pm25 = df[df["pollutant_id"] == "PM2.5"]
pm10 = df[df["pollutant_id"] == "PM10"]
print("PM2.5 rows:", len(pm25))
print("PM10  rows:", len(pm10))

print("\n=== Top 10 States by PM2.5 ===")
state_pm25 = pm25.groupby("state")["pollutant_avg"].mean().sort_values(ascending=False)
print(state_pm25.head(10))

print("\n=== Top 10 Cities by PM2.5 ===")
city_pm25 = pm25.groupby("city")["pollutant_avg"].mean().sort_values(ascending=False)
print(city_pm25.head(10))

# How many PM2.5 readings are above 90 (Poor category)
above_90 = pm25[pm25["pollutant_avg"] > 90]
print("\nReadings above 90 :", len(above_90))
pct = len(above_90) / len(pm25) * 100
print("Percentage        : {:.1f}%".format(pct))

# Statistical Analysis
print("\n=== Statistics for PM2.5 and PM10 ===")
pm25_vals = pm25["pollutant_avg"].dropna()
pm10_vals = pm10["pollutant_avg"].dropna()

print("PM2.5 - mean:", round(pm25_vals.mean(), 2), "median:", round(pm25_vals.median(), 2), "std:", round(pm25_vals.std(), 2))
print("PM10  - mean:", round(pm10_vals.mean(),  2), "median:", round(pm10_vals.median(),  2), "std:", round(pm10_vals.std(),  2))
print("PM2.5 skewness:", round(pm25_vals.skew(), 2))
print("PM10  skewness:", round(pm10_vals.skew(),  2))

# Pearson correlation
pm25_df = df[df["pollutant_id"] == "PM2.5"][["station", "pollutant_avg"]].rename(columns={"pollutant_avg": "PM2.5"})
pm10_df = df[df["pollutant_id"] == "PM10"][["station",  "pollutant_avg"]].rename(columns={"pollutant_avg": "PM10"})
merged = pm25_df.merge(pm10_df, on="station")

print("\nStations with both readings:", len(merged))
r, p = stats.pearsonr(merged["PM2.5"], merged["PM10"])
print("Pearson r:", round(r, 3), "  p-value:", round(p, 4))

## 4. Data Visualization

This code matches `Python/data_visualization.py` exactly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../Dataset/cleaned dataset.csv")
pm25 = df[df["pollutant_id"] == "PM2.5"]
pm10 = df[df["pollutant_id"] == "PM10"]

# 1. Number of readings per pollutant category (Category Analysis)
plt.figure(figsize=(7, 4))
df["pollutant_id"].value_counts().plot(kind="barh", color="seagreen")
plt.title("Readings per pollutant category")
plt.xlabel("Count")
plt.tight_layout()
plt.savefig("../Visualizations/category_analysis.png", dpi=150)
print("Saved category_analysis.png")
plt.show()

# 2. PM2.5 histogram (Distribution Analysis)
plt.figure(figsize=(7, 4))
plt.hist(pm25["pollutant_avg"].dropna(), bins=30, color="indianred", edgecolor="white")
plt.title("PM2.5 distribution")
plt.xlabel("Average reading")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("../Visualizations/distribution_analysis.png", dpi=150)
print("Saved distribution_analysis.png")
plt.show()

# 3. Top 10 states bar chart for PM2.5 (Trend Analysis)
top_states = pm25.groupby("state")["pollutant_avg"].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 5))
top_states.plot(kind="bar", color="firebrick")
plt.title("Top 10 States with highest PM2.5")
plt.ylabel("Average PM2.5")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../Visualizations/trend_analysis.png", dpi=150)
print("Saved trend_analysis.png")
plt.show()

# 4. Scatter plot PM2.5 vs PM10 (Correlation Analysis)
pm25_st = pm25.groupby("station")["pollutant_avg"].mean()
pm10_st = pm10.groupby("station")["pollutant_avg"].mean()
merged = pd.DataFrame({"PM2.5": pm25_st, "PM10": pm10_st}).dropna()

plt.figure(figsize=(7, 5))
plt.scatter(merged["PM10"], merged["PM2.5"], alpha=0.6, color="purple")
plt.title("PM10 vs PM2.5 Averages per Station")
plt.xlabel("PM10 Average")
plt.ylabel("PM2.5 Average")
plt.tight_layout()
plt.savefig("../Visualizations/correlation_analysis.png", dpi=150)
print("Saved correlation_analysis.png")
plt.show()